# Step 2: Data Preprocessing
## Attention-CNN-LSTM for Intrusion Detection (Bot-IoT Dataset)

This notebook will:
1. Load and merge all raw CSV files
2. Clean the data (handle missing values, drop irrelevant columns)
3. Encode categorical features
4. Apply Z-score normalization (as described in the paper)
5. Split into 80% train / 20% test
6. Save processed data for the model notebook

> **Make sure you completed Notebook 1 before running this.**

---
## Cell 1: Imports

In [ ]:
import pandas as pd
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

print("✅ Imports successful")

✅ Imports successful


---
## Cell 2: Load Dataset from Parquet

In [13]:
parquet_path = '../data/processed/botiot_raw.parquet'

assert os.path.exists(parquet_path), (
    f"Parquet file not found at '{parquet_path}'. "
    "Run notebook 01_setup_and_download.ipynb first to generate it."
)

df = pd.read_parquet(parquet_path, engine='fastparquet')

print(f"✅ Loaded dataset from parquet: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nColumns: {list(df.columns)}")

✅ Loaded dataset from parquet: 4,000,000 rows × 35 columns

Columns: ['pkSeqID', 'stime', 'flgs', 'proto', 'saddr', 'sport', 'daddr', 'dport', 'pkts', 'bytes', 'state', 'ltime', 'seq', 'dur', 'mean', 'stddev', 'smac', 'dmac', 'sum', 'min', 'max', 'soui', 'doui', 'sco', 'dco', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'srate', 'drate', 'attack', 'category', 'subcategory']


---
## Cell 3: Explore and Clean

The Bot-IoT dataset has some columns that are not useful for ML:
- IP addresses (saddr, daddr) — too specific, cause overfitting
- Timestamps — raw timestamps are not useful directly
- Category + subcategory — we use only 'category' as our label

In [14]:
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(list(df.columns))

print("\nData types:")
print(df.dtypes.value_counts())

Dataset shape: (4000000, 35)

Column names:
['pkSeqID', 'stime', 'flgs', 'proto', 'saddr', 'sport', 'daddr', 'dport', 'pkts', 'bytes', 'state', 'ltime', 'seq', 'dur', 'mean', 'stddev', 'smac', 'dmac', 'sum', 'min', 'max', 'soui', 'doui', 'sco', 'dco', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'srate', 'drate', 'attack', 'category', 'subcategory']

Data types:
float64    19
int64       9
string      7
Name: count, dtype: int64


In [15]:
# -------------------------------------------------------
# LABEL COLUMN SETUP
# Bot-IoT uses 'category' as the main attack type label
# -------------------------------------------------------
LABEL_COL = 'category'  # Main attack category label

# Verify label column exists
assert LABEL_COL in df.columns, f"Column '{LABEL_COL}' not found! Check column names above."

print(f"Label column: '{LABEL_COL}'")
print(f"\nClass distribution:")
print(df[LABEL_COL].value_counts())
print(f"\nNumber of classes: {df[LABEL_COL].nunique()}")

Label column: 'category'

Class distribution:
category
DoS               2169635
Reconnaissance    1821639
Normal               7139
Theft                1587
Name: count, dtype: int64[pyarrow]

Number of classes: 4


In [16]:
# Drop columns that are not useful for model training
# These are identifiers / metadata, not features

cols_to_drop = []

# Drop IP addresses if they exist
for col in ['saddr', 'daddr', 'saddr_', 'daddr_']:
    if col in df.columns:
        cols_to_drop.append(col)

# Drop subcategory (too detailed; we use category as label)
for col in ['subcategory', 'attack', 'pkSeqID']:
    if col in df.columns:
        cols_to_drop.append(col)

if cols_to_drop:
    df.drop(columns=cols_to_drop, inplace=True)
    print(f"✅ Dropped {len(cols_to_drop)} columns: {cols_to_drop}")
else:
    print("No columns dropped (none of the target columns found)")

print(f"Remaining columns: {df.shape[1]}")

✅ Dropped 5 columns: ['saddr', 'daddr', 'subcategory', 'attack', 'pkSeqID']
Remaining columns: 30


In [17]:
# Handle missing values
missing = df.isnull().sum()
missing_cols = missing[missing > 0]

if len(missing_cols) > 0:
    print(f"Found missing values in {len(missing_cols)} columns:")
    print(missing_cols)
    # Fill numerical columns with median
    for col in missing_cols.index:
        if df[col].dtype in ['float64', 'int64']:
            df[col].fillna(df[col].median(), inplace=True)
        else:
            df[col].fillna(df[col].mode()[0], inplace=True)
    print("✅ Missing values filled")
else:
    print("✅ No missing values found")

Found missing values in 8 columns:
sport      32669
dport      32670
smac     4000000
dmac     4000000
soui     4000000
doui     4000000
sco      4000000
dco      4000000
dtype: int64
✅ Missing values filled


In [18]:
# Remove duplicate rows
before = len(df)
df.drop_duplicates(inplace=True)
after = len(df)
print(f"Removed {before - after:,} duplicate rows")
print(f"Remaining rows: {after:,}")

Removed 0 duplicate rows
Remaining rows: 4,000,000


---
## Cell 4: Handle Categorical Features

ML models need numbers. Any text columns (like protocol type) must be encoded.

In [19]:
# Identify categorical columns (non-numeric, excluding label)
categorical_cols = df.select_dtypes(exclude='number').columns.tolist()
categorical_cols = [c for c in categorical_cols if c != LABEL_COL]

print(f"Categorical feature columns found: {categorical_cols}")

# Encode each categorical column using LabelEncoder
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le
    print(f"  Encoded '{col}': {le.classes_[:5]} → [0, 1, 2, ...]")

print("\n✅ Categorical features encoded")

Categorical feature columns found: ['flgs', 'proto', 'state']
  Encoded 'flgs': ['e' 'e    F' 'e   t' 'e  D' 'e &'] → [0, 1, 2, ...]
  Encoded 'proto': ['arp' 'icmp' 'igmp' 'ipv6-icmp' 'rarp'] → [0, 1, 2, ...]
  Encoded 'state': ['ACC' 'CLO' 'CON' 'ECO' 'FIN'] → [0, 1, 2, ...]

✅ Categorical features encoded


In [20]:
# Encode the TARGET label column
label_encoder_target = LabelEncoder()
df['label_encoded'] = label_encoder_target.fit_transform(df[LABEL_COL])

print("Label encoding mapping:")
for i, cls in enumerate(label_encoder_target.classes_):
    count = (df['label_encoded'] == i).sum()
    print(f"  {i} → '{cls}' ({count:,} samples)")

NUM_CLASSES = len(label_encoder_target.classes_)
print(f"\nTotal classes: {NUM_CLASSES}")

Label encoding mapping:
  0 → 'DoS' (2,169,635 samples)
  1 → 'Normal' (7,139 samples)
  2 → 'Reconnaissance' (1,821,639 samples)
  3 → 'Theft' (1,587 samples)

Total classes: 4


---
## Cell 5: Feature Selection

Separate features (X) from labels (y).

In [21]:
# Separate features and labels
drop_for_X = [LABEL_COL, 'label_encoded']
feature_cols = [c for c in df.columns if c not in drop_for_X]

X = df[feature_cols].values.astype(np.float32)
y = df['label_encoded'].values.astype(np.int64)

print(f"Feature matrix X shape : {X.shape}")
print(f"Label vector y shape   : {y.shape}")
print(f"Number of features     : {X.shape[1]}")
print(f"Number of classes      : {NUM_CLASSES}")
print(f"\nFeature columns:")
for i, col in enumerate(feature_cols):
    print(f"  {i+1:2d}. {col}")

Feature matrix X shape : (4000000, 29)
Label vector y shape   : (4000000,)
Number of features     : 29
Number of classes      : 4

Feature columns:
   1. stime
   2. flgs
   3. proto
   4. sport
   5. dport
   6. pkts
   7. bytes
   8. state
   9. ltime
  10. seq
  11. dur
  12. mean
  13. stddev
  14. smac
  15. dmac
  16. sum
  17. min
  18. max
  19. soui
  20. doui
  21. sco
  22. dco
  23. spkts
  24. dpkts
  25. sbytes
  26. dbytes
  27. rate
  28. srate
  29. drate


---
## Cell 6: Z-Score Normalization

From the paper (Equation 1): **x' = (x - μ) / σ**

This ensures all features are on the same scale so no single feature dominates training.

In [22]:
# Apply Z-score normalization (StandardScaler = Z-score)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Before normalization (first feature):")
print(f"  Mean: {X[:, 0].mean():.4f}, Std: {X[:, 0].std():.4f}")
print(f"  Min:  {X[:, 0].min():.4f}, Max: {X[:, 0].max():.4f}")

print("\nAfter normalization (first feature):")
print(f"  Mean: {X_scaled[:, 0].mean():.4f}, Std: {X_scaled[:, 0].std():.4f}")
print(f"  Min:  {X_scaled[:, 0].min():.4f}, Max: {X_scaled[:, 0].max():.4f}")

print("\n✅ Z-score normalization applied")

Before normalization (first feature):
  Mean: 1527533568.0000, Std: 608526.6875
  Min:  1526344064.0000, Max: 1529381888.0000

After normalization (first feature):
  Mean: 0.0001, Std: 1.0000
  Min:  -1.9545, Max: 3.0376

✅ Z-score normalization applied


In [23]:
# Replace inf and nan values that may appear after normalization
X_scaled = np.nan_to_num(X_scaled, nan=0.0, posinf=0.0, neginf=0.0)
print(f"NaN values after scaling: {np.isnan(X_scaled).sum()}")
print(f"Inf values after scaling: {np.isinf(X_scaled).sum()}")
print("✅ Data is clean")

NaN values after scaling: 0
Inf values after scaling: 0
✅ Data is clean


---
## Cell 7: Train/Test Split

From the paper: **80% training, 20% testing** with stratified sampling to maintain class balance.

In [24]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # Maintains class distribution in both splits
)

print(f"Training set   : {X_train.shape[0]:,} samples ({X_train.shape[0]/len(y)*100:.1f}%)")
print(f"Test set       : {X_test.shape[0]:,} samples ({X_test.shape[0]/len(y)*100:.1f}%)")
print(f"Feature count  : {X_train.shape[1]}")

print("\nClass distribution in training set:")
unique, counts = np.unique(y_train, return_counts=True)
for cls_id, cnt in zip(unique, counts):
    cls_name = label_encoder_target.classes_[cls_id]
    print(f"  {cls_id} ({cls_name:20s}): {cnt:,}")

Training set   : 3,200,000 samples (80.0%)
Test set       : 800,000 samples (20.0%)
Feature count  : 29

Class distribution in training set:
  0 (DoS                 ): 1,735,708
  1 (Normal              ): 5,711
  2 (Reconnaissance      ): 1,457,311
  3 (Theft               ): 1,270


---
## Cell 8: Reshape for CNN-LSTM Input

CNN and LSTM expect **3D input**: `(samples, timesteps, features)`

Since each row is one network record (not a time series), we treat it as:
- **timesteps = 1** (each record is one timestep)
- **features = number of columns**

This is the standard approach for tabular data fed into CNN-LSTM.

In [25]:
# Reshape: (samples, features) → (samples, 1, features)
X_train_3d = X_train.reshape(X_train.shape[0], 1, X_train.shape[1])
X_test_3d  = X_test.reshape(X_test.shape[0],  1, X_test.shape[1])

print(f"X_train shape: {X_train.shape} → {X_train_3d.shape}")
print(f"X_test shape : {X_test.shape}  → {X_test_3d.shape}")
print(f"\nDimensions: (samples, timesteps, features)")
print("✅ Data reshaped for CNN-LSTM input")

X_train shape: (3200000, 29) → (3200000, 1, 29)
X_test shape : (800000, 29)  → (800000, 1, 29)

Dimensions: (samples, timesteps, features)
✅ Data reshaped for CNN-LSTM input


---
## Cell 9: Save Processed Data

In [26]:
import pickle

# Save numpy arrays
np.save('../data/processed/X_train.npy', X_train_3d)
np.save('../data/processed/X_test.npy',  X_test_3d)
np.save('../data/processed/y_train.npy', y_train)
np.save('../data/processed/y_test.npy',  y_test)

# Save scaler and label encoder for later use
with open('../data/processed/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

with open('../data/processed/label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder_target, f)

# Save metadata
metadata = {
    'num_features': X_train.shape[1],
    'num_classes': NUM_CLASSES,
    'class_names': list(label_encoder_target.classes_),
    'train_samples': X_train.shape[0],
    'test_samples': X_test.shape[0],
    'feature_cols': feature_cols
}

with open('../data/processed/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("✅ Saved:")
print("  ../data/processed/X_train.npy")
print("  ../data/processed/X_test.npy")
print("  ../data/processed/y_train.npy")
print("  ../data/processed/y_test.npy")
print("  ../data/processed/scaler.pkl")
print("  ../data/processed/label_encoder.pkl")
print("  ../data/processed/metadata.json")
print("\n" + "="*50)
print("PREPROCESSING COMPLETE! Ready for Notebook 3.")
print("="*50)
print(f"""
Summary:
  ✅ {X_train.shape[0]:,} training samples
  ✅ {X_test.shape[0]:,} test samples
  ✅ {X_train.shape[1]} features
  ✅ {NUM_CLASSES} classes: {list(label_encoder_target.classes_)}
  ✅ Z-score normalized
  ✅ Reshaped to (samples, 1, features)

Next step: Run notebook → 03_model_training.ipynb
""")

✅ Saved:
  data/processed/X_train.npy
  data/processed/X_test.npy
  data/processed/y_train.npy
  data/processed/y_test.npy
  data/processed/scaler.pkl
  data/processed/label_encoder.pkl
  data/processed/metadata.json

PREPROCESSING COMPLETE! Ready for Notebook 3.

Summary:
  ✅ 3,200,000 training samples
  ✅ 800,000 test samples
  ✅ 29 features
  ✅ 4 classes: ['DoS', 'Normal', 'Reconnaissance', 'Theft']
  ✅ Z-score normalized
  ✅ Reshaped to (samples, 1, features)

Next step: Run notebook → 03_model_training.ipynb



In [ ]:
import gc

for var in ['df', 'X', 'X_scaled', 'X_train', 'X_test', 'X_train_3d', 'X_test_3d', 'y', 'y_train', 'y_test', 'dfs']:
    if var in dir():
        del var

gc.collect()

# Release all cached GPU memory
torch.cuda.empty_cache()

print("✅ Memory cleared.")

✅ Memory cleared.
